# Processing - Raw Usage Data.ipynb

## Overview

This notebook loads **raw metering records** from `data/raw/` and produces interactive
charts for auditing, debugging, and understanding instance-level usage patterns.

**Raw data** means individual, time-stamped usage records — one row per
`(instance, metric, time-window)` pair. It is the most granular data available from the
Sovereign Core metrics-aggregator API and is the right starting point when you need to:

- **Audit** that all expected instances and tenants are reporting
- **Debug** a specific instance's usage history
- **Understand reporting patterns** — which metrics fire how often, and from how many instances
- **Identify outliers** — instances with unusually high or absent usage

If you are looking to understand trends over time or compare tenants side-by-side,
use **`Processing - Aggregated Usage Data.ipynb`** or **`Processing - Grouped Usage Data.ipynb`** instead.

---

### What this notebook produces

| # | Chart | What it answers |
|---|---|---|
| 5.1 | Daily usage per metric — line chart | How is each metric trending over the window? |
| 5.2 | Reporting frequency heatmap | Which tenants are consistently reporting vs. going silent? |
| 5.3 | Daily usage breakdown — stacked bar | What share of total usage does each metric contribute per day? |
| 5.4 | Active instances per day | How many distinct instances reported on each day? |
| 5.5 | Per-instance peak usage — top 10 | Which instances are the heaviest consumers per metric? |
| 5.6 | Metering model distribution — donut | What mix of `point-in-time`, `total-up-to-date`, and `high-watermark` metrics exist? |
| 5.7 | Service coverage — KPI cards | How many distinct instances, tenants, workspaces, and regions are active? |

---

### Prerequisites

- This notebook runs out of the box using the **included sample data** — no deployment or API access needed.
- To use your own data, run **`Fetch - Usage Data.ipynb`** first to populate `data/raw/<APP_DOMAIN>/<SERVICE_ID>/`, or point the configuration (Section 2) at an existing data directory.

All charts are fully interactive — hover for exact values, click legend items to toggle series,
drag to zoom, double-click to reset.


## 1. Imports

In [1]:
import json
import math
import os
import pathlib

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

## 2. Configuration

Load data path from environment variables or use defaults.
If you don't set `RAW_DATA_PATH` in `.env`, we'll use `data/raw/<APP_DOMAIN>/<SERVICE_ID>/`.
To use custom data, edit `.env` and set `RAW_DATA_PATH`.

No secrets are loaded or printed here.

In [56]:
if pathlib.Path(".env").exists():
    load_dotenv(".env")
else:
    load_dotenv(".env.template")

service_id  = os.getenv("SERVICE_ID", "").strip()
app_domain  = os.getenv("APP_DOMAIN", "").strip()

# Use nested path structure: data/raw/{APP_DOMAIN}/{SERVICE_ID}/
# APP_DOMAIN and SERVICE_ID are mandatory config values
if os.getenv("RAW_DATA_PATH"):
    data_path = os.getenv("RAW_DATA_PATH")  # Allow override via env var
else:
    data_path = f"data/raw/{app_domain}/{service_id}"

DATA_DIR    = pathlib.Path(data_path)

json_files = list(DATA_DIR.glob("*.json"))

if json_files:
    print(f"✓ Using data path: {data_path}")
    print(f"✓ Found {len(json_files)} JSON file(s)")
else:
    print(f"⚠ No JSON files found in: {DATA_DIR.resolve()}")
    print(f"\nTo use a different path, edit .env and set:")
    print(f"  RAW_DATA_PATH=/path/to/your/data")
    print(f"\nExample: RAW_DATA_PATH=sample_data/raw")

✓ Using data path: data/raw/apps.gori-agent-hub.cp.fyre.ibm.com/servicebrokercore
✓ Found 5 JSON file(s)


## 3. Load Raw Data

### Data format

Each JSON file has the shape returned by `GET /metering/services/{serviceId}/usage/raw`:

```json
{
  "params": { ... },
  "meteredUsage": [
    {
      "id": 42,
      "instanceId": "...",
      "tenantId": "...",
      "workspaceId": "...",
      "serviceId": "...",
      "region": "...",
      "metricId": "users",
      "meteringModel": "total-up-to-date",
      "usageQuantity": 7.0,
      "startTimestamp": "2026-05-10T00:00:00Z",
      "endTimestamp": "2026-05-10T01:00:00Z",
      "correlationId": "...",
      "transactionId": "..."
    }
  ]
}
```

All files are merged into a single DataFrame. Duplicate `id` values are dropped — keeping the first occurrence.

In [ ]:
records = []
for path in sorted(DATA_DIR.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    records.extend(data.get("meteredUsage", []))

df = pd.DataFrame(records)

if df.empty:
    print("⚠ No records found in the fetched data.")
    print("Possible reasons:")
    print("  • The time period has no usage data")
    print("  • The service ID or tenant ID filters returned no matches")
    print("  • The API credentials lack access to this data")
    print("\nSkipping data processing. Please check your configuration and try again.")
else:
    df.drop_duplicates(subset="id", keep="first", inplace=True)
    df["startTimestamp"] = pd.to_datetime(df["startTimestamp"], format="mixed")
    df["endTimestamp"]   = pd.to_datetime(df["endTimestamp"],   format="mixed")
    df["usageQuantity"]  = pd.to_numeric(df["usageQuantity"])
    df["tenantLabel"]    = df["tenantId"].replace("", "(unset)")
    df["date"]           = df["startTimestamp"].dt.floor("D")

    metrics  = sorted(df["metricId"].unique())
    tenants  = sorted(df["tenantLabel"].unique())

    print(f"Total records : {len(df)}")
    print(f"Date range    : {df['startTimestamp'].min()} → {df['startTimestamp'].max()}")
    print(f"Metrics       : {metrics}")
    print(f"Tenants       : {tenants}")
    print(f"Instances     : {df['instanceId'].nunique()} unique")
    df.head(3)

Total records : 36394
Date range    : 2026-08-11 14:23:03.436000+00:00 → 2026-08-24 05:45:31.162000+00:00
Metrics       : ['api_calls', 'instances', 'users']
Tenants       : ['0fd2a329-b354-44ad-90e7-16f00fa85948', '7ef3d047-2729-402c-a956-3292c88329a7', 'ab475c00-63cc-4b75-abcf-9c4513505e31', 'platform']
Instances     : 8 unique


## 4. Summary table — usage per tenant × metric

| Metering model | Value shown | Rationale |
|---|---|---|
| `point-in-time`    | **avg, min, max** | Snapshots fluctuate — mean, min or max can be the representative level |
| `total-up-to-date` | **sum** | Consumption per time interval - Total usage / count |
| `high-watermark`   | **avg** | Maximum observed value — avg gives the mean usage |

**Note :**
calculations made here are for a highlevel over view 

In [ ]:
AGGS = {
    "point-in-time":    ("avg", "mean"),
    "point-in-time":    ("max", "max"),
    "point-in-time":    ("min", "min"),
    "total-up-to-date": ("avg", "mean"),
    "high-watermark":   ("max", "max"),
}

rows = []
for (tenant, metric), grp in df.groupby(["tenantLabel", "metricId"]):
    model = grp["meteringModel"].iloc[0]
    label, fn = AGGS.get(model, ("avg", "mean"))
    value = getattr(grp["usageQuantity"], fn)()
    rows.append({
        "Tenant":          tenant,
        "Metric":          metric,
        "Metering model":  model,
        "Value (agg)":     label,
        "Usage quantity":  round(value, 3),
        "Instances":       grp["instanceId"].nunique(),
        "Workspaces":      grp["workspaceId"].nunique(),
    })

summary = pd.DataFrame(rows).sort_values(["Tenant", "Metric"]).reset_index(drop=True)
summary

,Tenant,Metric,Metering model,Value (agg),Usage quantity,Instances,Workspaces
0,0fd2a329-b354-44ad-90e7-16f00fa85948,api_calls,total-up-to-date,avg,498.819,5,3
1,0fd2a329-b354-44ad-90e7-16f00fa85948,instances,point-in-time,min,1.000,5,3
2,0fd2a329-b354-44ad-90e7-16f00fa85948,users,point-in-time,min,1.000,5,3
3,7ef3d047-2729-402c-a956-3292c88329a7,api_calls,total-up-to-date,avg,499.494,1,1
4,7ef3d047-2729-402c-a956-3292c88329a7,instances,point-in-time,min,1.000,1,1
5,7ef3d047-2729-402c-a956-3292c88329a7,users,point-in-time,min,1.000,1,1
6,ab475c00-63cc-4b75-abcf-9c4513505e31,api_calls,total-up-to-date,avg,500.654,1,1
7,ab475c00-63cc-4b75-abcf-9c4513505e31,instances,point-in-time,min,1.000,1,1
8,ab475c00-63cc-4b75-abcf-9c4513505e31,users,point-in-time,min,1.000,1,1
9,platform,api_calls,total-up-to-date,avg,498.684,1,1


## 5. Charts


### 5.1 Daily usage per metric — line chart

### Computation

Groups records by `(metricId, meteringModel, date)` and applies the correct aggregation per model:
- `total-up-to-date` → **sum** per day (each submission is incremental consumption)
- `point-in-time` → **mean** per day (each submission is a snapshot; averaging avoids double-counting)
- `high-watermark` → **mean** per day (records represent the observed peak; averaging gives typical load)

Each series is labelled `metricId [meteringModel]` so the legend is self-explanatory.
The Y axis title reflects the aggregation logic in use.


In [59]:
def _model_aware_daily(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a daily aggregated value per (metricId, meteringModel) that is
    semantically correct for each metering model.
    """
    parts = []
    for (metric, model), grp in frame.groupby(["metricId", "meteringModel"]):
        if model == "total-up-to-date":
            agg = grp.groupby("date")["usageQuantity"].sum().reset_index()
            y_label = "Daily total usage (sum of submissions)"
        elif model == "point-in-time":
            agg = grp.groupby("date")["usageQuantity"].mean().reset_index()
            y_label = "Daily usage (average of snapshots)"
        else:  # high-watermark
            agg = grp.groupby("date")["usageQuantity"].mean().reset_index()
            y_label = "Daily avg peak reading"
        agg["metricId"] = metric
        agg["meteringModel"] = model
        agg["yLabel"] = y_label
        parts.append(agg)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=["date", "usageQuantity", "metricId", "meteringModel", "yLabel"])


daily = _model_aware_daily(df)
daily["seriesLabel"] = daily.apply(
    lambda r: f"{r['metricId']} [{r['meteringModel']}]", axis=1)

y_titles = daily[["meteringModel", "yLabel"]].drop_duplicates()
y_axis_title = (
    y_titles.iloc[0]["yLabel"]
    if len(y_titles) == 1
    else "Usage quantity"   # short label when multiple models are mixed
)

n_series    = daily["seriesLabel"].nunique()
legend_rows = math.ceil(n_series / 2)   # ~2 items per row
legend_b    = 80 + legend_rows * 24     # enough gap between tick labels and legend
chart_h     = 420 + legend_rows * 20    # chart body grows so legend does not compress it

fig_51 = px.line(
    daily, x="date", y="usageQuantity", color="seriesLabel",
    markers=True,
    title="Daily usage per metric (model-aware aggregation)",
    labels={"date": "Date", "usageQuantity": y_axis_title, "seriesLabel": "Metric"},
)
fig_51.update_layout(
    hovermode="x unified",
    plot_bgcolor="white",
    height=chart_h,
    margin=dict(t=80, b=legend_b, l=80, r=40),
    legend=dict(
        title="Metric",
        orientation="h",
        yanchor="top", y=-0.22,
        xanchor="left", x=0,
    ),
);


### Chart Guide

**Purpose:** Shows how each metric is trending day-over-day across the query window,
using the semantically correct aggregation for each metering model.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Multi-series line chart |
| **X axis** | Date (one data point per day) |
| **Y axis** | Usage quantity — aggregation varies by metering model |
| **Colour** | One line per metric, labelled with its metering model |

**Insights:** A rising line means consumption is growing. A flat line means stable usage.
Sudden drops may indicate reporting gaps or instance churn rather than real usage decreases.


In [ ]:
if fig_51 is not None:
    fig_51.show()


### 5.2 Metric reporting frequency by tenant — heatmap

### Computation

Groups the raw records by `(tenantLabel, metricId)` and counts the number of rows in each group.
The result is pivoted into a matrix (tenants × metrics) with missing combinations filled as zero.
The raw count is printed inside each cell for quick reading without hovering.


In [61]:
counts = (
    df.groupby(["tenantLabel", "metricId"])
    .size()
    .reset_index(name="count")
)
pivot = counts.pivot(index="tenantLabel", columns="metricId", values="count").fillna(0)

fig_52 = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Metric reporting frequency by tenant",
    labels={"x": "Metric", "y": "Tenant", "color": "Record count"},
    aspect="auto",
)
fig_52.update_layout(coloraxis_colorbar_title="Records");


### Chart Guide

**Purpose:** Reveals which tenants are actively reporting each metric and which are silent,
making it easy to spot misconfigured metering agents at a glance.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Annotated heatmap |
| **X axis** | Metric ID |
| **Y axis** | Tenant (short label) |
| **Colour** | Record count — darker blue = more submissions |

**Insights:** A completely white row means a tenant has submitted no data for that metric.
Uneven columns suggest some metrics are emitted less frequently than others by design.


In [ ]:
if fig_52 is not None:
    fig_52.show()


### 5.3 Daily usage breakdown by metric — stacked bar

### Computation

Reuses the `daily` DataFrame from section 5.1 — no additional aggregation.
Dates are formatted as `Mon DD` strings so bars are evenly spaced regardless of calendar gaps.
The `category_orders` argument preserves strict chronological ordering on the X axis.


In [63]:
daily_bar = daily.copy()
daily_bar["dateLabel"] = daily_bar["date"].dt.strftime("%b %d")

fig_53 = px.bar(
    daily_bar, x="dateLabel", y="usageQuantity", color="seriesLabel",
    barmode="stack",
    title="Daily usage by metric — stacked bar",
    labels={"dateLabel": "Date", "usageQuantity": y_axis_title, "seriesLabel": "Metric"},
    category_orders={"dateLabel": daily_bar.drop_duplicates("date").sort_values("date")["dateLabel"].tolist()},
)
n_series  = daily_bar["seriesLabel"].nunique()
legend_rows = math.ceil(n_series / 2)
legend_b  = 80 + legend_rows * 24     # gap clears rotated tick labels before legend starts
chart_h   = 420 + legend_rows * 20    # chart body grows so legend does not compress it
legend_y  = -(legend_b / chart_h)     # dynamic offset keeps legend below tick labels at any scale
fig_53.update_layout(
    bargap=0.2,
    hovermode="x unified",
    plot_bgcolor="white",
    height=chart_h,
    xaxis=dict(tickangle=-45, tickmode="linear"),
    margin=dict(t=80, b=legend_b, l=80, r=40),
    legend=dict(
        title="Metric",
        orientation="h",
        yanchor="top", y=legend_y,
        xanchor="left", x=0,
    ),
);


### Chart Guide

**Purpose:** Shows daily total usage and how each metric contributes to it.
Where the line chart (5.1) shows individual metric trends, this chart shows relative share.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Stacked bar chart |
| **X axis** | Date |
| **Y axis** | Usage quantity (same model-aware aggregation as 5.1) |
| **Colour** | One segment per metric |

**Insights:** A metric whose segment grows relative to others is increasing its share.
A very thin segment means that metric contributes negligibly to overall load.


In [ ]:
if fig_53 is not None:
    fig_53.show()


### 5.4 Active instances per day — bar chart

### Computation

Groups raw records by `date` and counts the number of distinct `instanceId` values per day
using `nunique()`. The result is one row per day with the count of unique instances that
submitted at least one record on that day.


In [65]:
active = df.groupby("date")["instanceId"].nunique().reset_index()
active.columns = ["date", "active_instances"]
active["dateLabel"] = active["date"].dt.strftime("%b %d")

fig_54 = px.bar(
    active, x="dateLabel", y="active_instances",
    title="Active instances per day",
    labels={"dateLabel": "Date", "active_instances": "Distinct instance count"},
    text="active_instances",
    category_orders={"dateLabel": active.sort_values("date")["dateLabel"].tolist()},
)
fig_54.update_traces(textposition="outside")
fig_54.update_layout(
    bargap=0.3,
    xaxis=dict(tickangle=-45, tickmode="linear"),
);


### Chart Guide

**Purpose:** Tracks how many instances were actively metering each day — a fleet health indicator.
A stable count means the instance population is consistent across the window.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Vertical bar chart |
| **X axis** | Date |
| **Y axis** | Count of distinct instance IDs reporting that day |
| **Label** | Raw count printed above each bar |

**Insights:** A sudden drop may indicate provisioning failures or network issues.
A gradual increase signals fleet growth.


In [ ]:
if fig_54 is not None:
    fig_54.show()


### 5.5 Per-instance peak usage — faceted horizontal bar

### Computation

Groups by `(metricId, instanceId)` and takes the maximum `usageQuantity` for each pair —
the peak value regardless of metering model. Within each metric, the top 10 instances
by peak value are selected and sorted ascending so the longest bar appears at the top.
All metrics are shown in a single figure using `make_subplots` with one subplot per metric
and independent X axes so metrics with different units remain readable.


In [67]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

peak_all = (
    df.groupby(["metricId", "instanceId"])["usageQuantity"]
    .max().reset_index()
    .rename(columns={"usageQuantity": "peakUsage"})
)
_model_for_metric = df.groupby("metricId")["meteringModel"].first().to_dict()
peak_all["meteringModel"] = peak_all["metricId"].map(_model_for_metric)

metrics_list = sorted(peak_all["metricId"].unique())
n_metrics = len(metrics_list)
subplot_height = 340

fig_55 = make_subplots(
    rows=n_metrics, cols=1,
    subplot_titles=[f"{m} [{_model_for_metric.get(m, '')}]" for m in metrics_list],
    shared_xaxes=False,
    vertical_spacing=0.18 / max(n_metrics, 1),
)

COLOURS = ["#636EFA","#EF553B","#00CC96","#AB63FA","#FFA15A","#19D3F3"]

for row_idx, metric_id in enumerate(metrics_list, start=1):
    top10 = (
        peak_all[peak_all["metricId"] == metric_id]
        .nlargest(10, "peakUsage")
        .sort_values("peakUsage")  # ascending so top bar is at top
    )
    top10 = top10.copy()
    # Use full UUID as the y key so bars are always distinct, then override tick
    # labels to show only the first 8 chars. Hover still shows the full UUID via customdata.
    short_labels = top10["instanceId"].str[:8] + "…"
    fig_55.add_trace(
        go.Bar(
            x=top10["peakUsage"], y=top10["instanceId"],
            orientation="h",
            customdata=top10[["instanceId"]].values,
            text=top10["peakUsage"].apply(lambda v: f"{v:,.0f}"),
            textposition="outside",
            marker_color=COLOURS[(row_idx - 1) % len(COLOURS)],
            hovertemplate="<b>%{customdata[0]}</b><br>Peak: %{x:,.0f}<extra></extra>",
            showlegend=False,
        ),
        row=row_idx, col=1,
    )
    yaxis_key = "yaxis" if row_idx == 1 else f"yaxis{row_idx}"
    fig_55.layout[yaxis_key].update(
        tickvals=top10["instanceId"].tolist(),
        ticktext=short_labels.tolist(),
    )

fig_55.update_layout(
    height=subplot_height * n_metrics,
    title_text="Per-instance peak usage — top 10 per metric",
    paper_bgcolor="white",
    margin=dict(l=20, r=80, t=80, b=40),
)
fig_55.update_xaxes(showgrid=True, gridcolor="#eee");


### Chart Guide

**Purpose:** Identifies the heaviest-consuming instances for each metric.
Each metric has its own subplot with an independent X axis so different units remain readable.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Faceted horizontal bar (one subplot per metric) |
| **X axis** | Peak usage value — independent scale per metric |
| **Y axis** | Instance ID (top 10 by peak value) |
| **Colour** | One colour per metric subplot |

**Insights:** A single instance dominating the top bar suggests uneven load distribution.
If the same instance tops multiple metrics it is a strong candidate for investigation.


In [ ]:
if fig_55 is not None:
    fig_55.show()


### 5.6 Metering model distribution — donut chart

### Computation

Counts the number of raw records per `meteringModel` value using `value_counts()`.
No date filtering or grouping — every record in the loaded dataset is counted.
The result is passed directly to `px.pie` with `hole=0.4` for the donut shape.


In [69]:
model_counts = df["meteringModel"].value_counts().reset_index()
model_counts.columns = ["meteringModel", "count"]

fig_56 = px.pie(
    model_counts, names="meteringModel", values="count",
    hole=0.4,
    title="Record distribution by metering model",
)
fig_56.update_traces(textinfo="label+percent", hovertemplate="%{label}: %{value} records")
fig_56.update_layout(margin=dict(t=80, b=40, l=40, r=40));


### Chart Guide

**Purpose:** Confirms what mix of metering semantics the dataset contains — important for
choosing the right aggregation transforms in downstream notebooks.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Donut chart |
| **Segments** | One per distinct `meteringModel` value |
| **Size** | Proportional to number of raw records with that model |

**Insights:** A single dominant segment is normal — most services use one model.
Multiple segments mean different metrics follow different metering semantics.


In [ ]:
if fig_56 is not None:
    fig_56.show()


### 5.7 Service coverage — distinct entities

### Computation

Counts `nunique()` for each of the four dimension columns (`instanceId`, `tenantId`,
`workspaceId`, `region`) across all loaded raw records.
The `region` column is included only if present — it is absent for some services.
Results are collected into a small DataFrame and rendered as a horizontal bar.


In [71]:
_dim_cols = {
    "Instances":   "instanceId",
    "Tenants":     "tenantId",
    "Workspaces":  "workspaceId",
    "Regions":     "region",
}
coverage = pd.DataFrame([
    {"Dimension": label, "Distinct count": df[col].nunique()}
    for label, col in _dim_cols.items() if col in df.columns
])

fig_57 = px.bar(
    coverage, x="Distinct count", y="Dimension",
    orientation="h", text="Distinct count",
    title="Service coverage — distinct entities in raw data",
    labels={"Distinct count": "Unique count", "Dimension": ""},
    color="Dimension",
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig_57.update_traces(textposition="outside", cliponaxis=False)
fig_57.update_layout(
    showlegend=False,
    xaxis=dict(showgrid=True, gridcolor="#eee"),
    yaxis=dict(categoryorder="total ascending"),
    plot_bgcolor="white",
    margin=dict(l=10, r=60, t=50, b=40),
);


### Chart Guide

**Purpose:** Gives a quick read on the deployment footprint of this service —
how many unique instances, tenants, workspaces, and regions are actively reporting.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Horizontal bar chart |
| **X axis** | Count of distinct values |
| **Y axis** | Dimension (Instances, Tenants, Workspaces, Regions) |
| **Colour** | One colour per dimension |

**Insights:** A high instance count with a low tenant count means few tenants run many instances each.
A region count of 1 means no geographic distribution.


In [ ]:
if fig_57 is not None:
    fig_57.show()
